# ⚙️ Hyperparameter Search for Fusion Classifiers

This notebook is part of the article **“DCMC-Fusion: Data-Centric and Model-Centric Ensemble Learning for Audio Deepfake Detection”**, by **Dora Ballesteros and Carlos Guerra**.

Its purpose is to identify a **single set of hyperparameters for each fusion classifier**, which will remain fixed across all architecture combinations and values of **K**.

- **Input dataset:** `inference_probabilities.csv`, containing the probabilities generated by 20 trained models: five architectures with four spectral representations each.

- **Search configuration:** all **20 probability columns** are used, corresponding to **K = 5**. The `label` column provides the ground truth: **0 = natural** and **1 = synthetic**.

- **Data partition:** a stratified **70/30 training–test split** is applied with `random_state=42`. Hyperparameter selection uses only the training subset; the test subset is reserved for the subsequent experiments.

- **Fusion classifiers:** Logistic Regression, Random Forest, XGBoost, and SVM.

- **Search procedure:** `GridSearchCV` uses **five-fold cross-validation** and **macro F1** as the selection metric. `StandardScaler` is included within each pipeline and fitted using only the training data of each fold.

- **Output:** the selected hyperparameters are saved in **`best_hyperparameters.json`** and reused in the five experimental notebooks without repeating the search.

- **Subsequent experiments:** each classifier and its scaler are fitted separately for each architecture combination, while the selected hyperparameters and the training–test partition remain fixed.

In [1]:
# Install dependencies
%pip install numpy pandas scikit-learn xgboost --quiet

In [2]:
# Data handling and output
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

# Preprocessing and hyperparameter search
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV

# Fusion classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# Reproducibility
RANDOM_STATE = 42
TEST_SIZE = 0.30
CV_FOLDS = 5
SCORING = "f1_macro"

np.random.seed(RANDOM_STATE)

In [3]:
# Load inference probabilities
CSV_PATH = Path("inference_probabilities.csv")
df = pd.read_csv(CSV_PATH)

# Five architectures, each with four spectral representations
ARCHITECTURES = {
    "resnet50": [
        "resnet50_mel", "resnet50_log",
        "resnet50_dwt", "resnet50_cqt"
    ],
    "densenet121": [
        "densenet121_mel", "densenet121_log",
        "densenet121_dwt", "densenet121_cqt"
    ],
    "inception_v3": [
        "inception_v3_mel", "inception_v3_log",
        "inception_v3_dwt", "inception_v3_cqt"
    ],
    "resnet34": [
        "resnet34_mel", "resnet34_log",
        "resnet34_dwt", "resnet34_cqt"
    ],
    "vgg16": [
        "vgg16_mel", "vgg16_log",
        "vgg16_dwt", "vgg16_cqt"
    ]
}

FEATURE_COLUMNS = [
    column
    for columns in ARCHITECTURES.values()
    for column in columns
]

# Validate input columns
required_columns = ["audio", "label"] + FEATURE_COLUMNS
missing_columns = sorted(set(required_columns) - set(df.columns))

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

# Input features and ground truth
X = df[FEATURE_COLUMNS].to_numpy(dtype=float)
y = df["label"].to_numpy()

if not np.isfinite(X).all() or ((X < 0) | (X > 100)).any():
    raise ValueError("Probabilities must be finite and in [0, 100].")

if set(np.unique(y)) != {0, 1}:
    raise ValueError("Labels must contain 0 (natural) and 1 (synthetic).")

# Split row indices to preserve sample identities
train_indices, test_indices = train_test_split(
    np.arange(len(df)),
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

X_train = X[train_indices]
y_train = y[train_indices]

# The test subset is reserved and is not used in the search
print(f"Input features: {X.shape[1]}")
print(f"Training samples: {len(train_indices)}")
print(f"Reserved test samples: {len(test_indices)}")

display(
    pd.Series(y_train)
    .value_counts()
    .sort_index()
    .rename_axis("label")
    .to_frame("training_samples")
)

Input features: 20
Training samples: 1738
Reserved test samples: 746


,training_samples
label,
0,869
1,869


In [4]:
# Classifiers and hyperparameter grids
ML_MODELS = {
    "Logistic Regression": (
        LogisticRegression(
            max_iter=1000,
            random_state=RANDOM_STATE
        ),
        {
            "model__C": [0.01, 0.1, 1, 10],
            "model__penalty": ["l1", "l2"],
            "model__solver": ["liblinear", "saga"]
        }
    ),

    "Random Forest": (
        RandomForestClassifier(
            random_state=RANDOM_STATE,
            n_jobs=1
        ),
        {
            "model__n_estimators": [100, 300, 700],
            "model__max_depth": [3, 5, 10, None],
            "model__min_samples_split": [3, 5, 8]
        }
    ),

    "XGBoost": (
        XGBClassifier(
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            verbosity=0,
            n_jobs=1
        ),
        {
            "model__n_estimators": [100, 300, 700],
            "model__learning_rate": [0.01, 0.1, 1],
            "model__max_depth": [3, 5, 8]
        }
    ),

    "SVM": (
        SVC(
            probability=True,
            random_state=RANDOM_STATE
        ),
        {
            "model__C": [0.1, 1, 5, 10],
            "model__gamma": ["scale", "auto"],
            "model__kernel": ["rbf"]
        }
    )
}

In [5]:
# Search using only the training subset
best_hyperparameters = {}
search_summary = []

for name, (classifier, parameter_grid) in ML_MODELS.items():
    print(f"\nSearching hyperparameters: {name}", flush=True)

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("model", classifier)
    ])

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=parameter_grid,
        cv=CV_FOLDS,
        scoring=SCORING,
        n_jobs=-1,
        refit=True,
        error_score="raise"
    )

    search.fit(X_train, y_train)

    # Remove the pipeline prefix for subsequent classifier creation
    parameters = {
        key.removeprefix("model__"): value
        for key, value in search.best_params_.items()
    }

    best_hyperparameters[name] = parameters

    search_summary.append({
        "classifier": name,
        "best_cv_f1_macro": search.best_score_,
        "best_parameters": json.dumps(parameters, sort_keys=True)
    })

    print(f"Best parameters: {parameters}")
    print(f"Best CV macro F1: {search.best_score_:.4f}")

search_summary_df = pd.DataFrame(search_summary)
display(search_summary_df)


Searching hyperparameters: Logistic Regression
Best parameters: {'C': 0.1, 'penalty': 'l2', 'solver': 'saga'}
Best CV macro F1: 0.7212

Searching hyperparameters: Random Forest
Best parameters: {'max_depth': 5, 'min_samples_split': 5, 'n_estimators': 700}
Best CV macro F1: 0.7272

Searching hyperparameters: XGBoost
Best parameters: {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100}
Best CV macro F1: 0.7267

Searching hyperparameters: SVM
Best parameters: {'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}
Best CV macro F1: 0.7084


,classifier,best_cv_f1_macro,best_parameters
0,Logistic Regression,0.721243,"{""C"": 0.1, ""penalty"": ""l2"", ""solver"": ""saga""}"
1,Random Forest,0.727244,"{""max_depth"": 5, ""min_samples_split"": 5, ""n_es..."
2,XGBoost,0.726709,"{""learning_rate"": 0.01, ""max_depth"": 3, ""n_est..."
3,SVM,0.708448,"{""C"": 0.1, ""gamma"": ""scale"", ""kernel"": ""rbf""}"


In [6]:
# Save selected hyperparameters
with open("best_hyperparameters.json", "w", encoding="utf-8") as file:
    json.dump(best_hyperparameters, file, indent=4)

# Save search scores
search_summary_df.to_csv(
    "hyperparameter_search_summary.csv",
    index=False
)

# Save the split and input order for subsequent experiments
split_configuration = {
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "cv_folds": CV_FOLDS,
    "scoring": SCORING,
    "search_K": 5,
    "feature_columns": FEATURE_COLUMNS,
    "train_indices": train_indices.tolist(),
    "test_indices": test_indices.tolist(),
    "audio_order": df["audio"].astype(str).tolist(),
    "label_order": y.tolist()
}

with open("experiment_split.json", "w", encoding="utf-8") as file:
    json.dump(split_configuration, file, indent=4)

print("Saved: best_hyperparameters.json")
print("Saved: hyperparameter_search_summary.csv")
print("Saved: experiment_split.json")

Saved: best_hyperparameters.json
Saved: hyperparameter_search_summary.csv
Saved: experiment_split.json
